# Fine-tuning GPT-2 for text generation

This guide describes fine-tuning a GPT-2 model with the WikiText-2 dataset for causal language modeling using Kubeflow Trainer.

This guide is adapted from the HuggingFace causal language modeling task page: https://huggingface.co/docs/transformers/en/tasks/language_modeling

Pretrained GPT-2: https://huggingface.co/openai-community/gpt2

WikiText-2 dataset: https://huggingface.co/datasets/Salesforce/wikitext

# Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

Install dependencies

In [ ]:
!pip install "transformers[torch]"

# Define the HuggingFace training script

We need to wrap our training script into a function to create the Kubeflow TrainJob.

In [ ]:
def train_gpt2(model_name: str, block_size: int = 128, num_samples: int = 1000):
    import os

    from datasets import load_dataset
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        DataCollatorForLanguageModeling,
        TrainingArguments,
        Trainer,
    )

    import torch.distributed as dist

    # Initialize distributed environment
    _, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    dist.init_process_group(backend=backend)

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    print(
        "Distributed Training with WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}.".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Download the dataset and tokenizer
    dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=f"train[:{num_samples}]")

    dataset = dataset.train_test_split(test_size=0.2, shuffle=False)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # GPT-2 does not have a pad token by default, so we use the eos token
    tokenizer.pad_token = tokenizer.eos_token

    # Tokenize the dataset
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True)

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset["train"].column_names,
    )

    # Group texts into chunks of block_size
    def group_texts(examples):
        # Concatenate all texts
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated["input_ids"])
        # Drop the remainder that doesn't fill a full block
        total_length = (total_length // block_size) * block_size
        # Split into chunks of block_size
        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated.items()
        }
        # For causal LM, labels are the same as input_ids
        result["labels"] = result["input_ids"].copy()
        return result

    lm_dataset = tokenized_dataset.map(group_texts, batched=True)

    # Create the data collator for causal language modeling (mlm=False)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # Load the model
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))

    # Define training hyperparameters
    training_args = TrainingArguments(
        output_dir=model_name,
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        num_train_epochs=1,
        weight_decay=0.01,
        push_to_hub=False,
    )

    # Prepare trainer with configuration
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=lm_dataset["train"],
        eval_dataset=lm_dataset["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

In [ ]:
from kubeflow.trainer import TrainerClient, CustomTrainer

for r in TrainerClient().list_runtimes():
    print(f"Name: {r.name}, Framework: {r.trainer.framework}, Trainer Type: {r.trainer.trainer_type.value}")

In [ ]:
MODEL_NAME = "gpt2"
args = {
    "model_name": MODEL_NAME,
    "block_size": 128,
    "num_samples": 1000,
}

job_id = TrainerClient().train(
    trainer=CustomTrainer(
        func=train_gpt2,
        func_args=args,
        num_nodes=1,
        packages_to_install=["datasets", "transformers[torch]"],
        resources_per_node={
            "cpu": "2",
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes
            # "nvidia.com/gpu": 1,
        },
    ),
)

In [ ]:
# Train API generates a random TrainJob id.
job_id

# Check the TrainJob details

Use `list_jobs()` and `get_job()` APIs to get details about the created TrainJob and its steps.

In [ ]:
for job in TrainerClient().list_jobs():
    print(f"TrainJob: {job.name}, Status: {job.status}, Created at: {job.creation_timestamp}")

In [ ]:
# Wait for the running status.
TrainerClient().wait_for_job_status(name=job_id, status={"Running"})

In [ ]:
for c in TrainerClient().get_job(name=job_id).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}")

# Show the TrainJob logs

Use `get_job_logs()` API to retrieve the TrainJob logs.

In [ ]:
for logline in TrainerClient().get_job_logs(job_id, follow=True):
    print(logline)

# Clean up

To delete the TrainJob you can use the `delete_job()` API and pass the generated `job_id`.

In [ ]:
# _ = TrainerClient().delete_job(job_id)